In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
import pandas as pd
import bm25s
import Stemmer
import duckdb
import hashlib
import numpy as np


In [2]:
passages_hashed = pd.read_parquet("../data/quati/passages.parquet")
qrels = pd.read_parquet("../data/quati/rel_queries_passages.parquet")
queries = pd.read_parquet("../data/quati/queries.parquet")


In [3]:
stemmer = Stemmer.Stemmer("portuguese")


In [12]:
corpus_tokens = bm25s.tokenize(passages_hashed["passage"], stopwords="pt", stemmer=stemmer)

Split strings:   0%|          | 0/1000000 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1000000 [00:00<?, ?it/s]

In [13]:
retriever = bm25s.BM25(corpus=passages_hashed.to_dict(orient='records'))
retriever.index(corpus_tokens)


BM25S Count Tokens:   0%|          | 0/1000000 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1000000 [00:00<?, ?it/s]

In [ ]:
retriever.save('../data/index_bm25', corpus=passages_hashed.to_dict(orient='records'), corpus_name='quati_corpus')


In [4]:
retriever = bm25s.BM25.load('../data/index_bm25', corpus_name='quati_corpus')

In [5]:
queries.shape

(200, 2)

In [6]:
queries.columns

Index(['query_id', 'query'], dtype='str')

In [6]:
queries.iloc[0]

query_id                                                   0
query       Qual a maior característica da flora brasileira?
Name: 0, dtype: object

In [14]:
query_tokens = bm25s.tokenize(queries['query'], stemmer=stemmer)

Split strings:   0%|          | 0/200 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/200 [00:00<?, ?it/s]

In [30]:
results, scores = retriever.retrieve(query_tokens, k=10)


BM25S Retrieve:   0%|          | 0/200 [00:00<?, ?it/s]

In [31]:
results.shape

(200, 10)

In [32]:
results

array([[{'passage_id': 'clueweb22-pt0000-05-02747_0', 'passage': 'BIOMAS E FITOFISIONOMIAS BRASILEIROS: CARACTERÍSTICAS E EVOLUÇÃO DA FAUNA E FLORA - Klima Naturali™ Klima Home Biomas BIOMAS E FITOFISIONOMIAS BRASILEIROS: CARACTERÍSTICAS E EVOLUÇÃO DA FAUNA E FLORA BIOMAS E FITOFISIONOMIAS BRASILEIROS: CARACTERÍSTICAS E EVOLUÇÃO DA FAUNA E FLORA Tags Biomas Um bioma é um conjunto de tipos de vegetação que abrange grandes áreas contínuas, em escala regional, com flora e fauna similares, definida pelas condições físicas predominantes nas regiões. Esses aspectos climáticos, geográficos e litológicos (das rochas), por exemplo, fazem com que um bioma seja dotado de uma diversidade biológica singular, própria. No Brasil, os biomas existentes são (da maior extensão para a menor): a Amazônia, o cerrado, a Mata Atlântica, a Caatinga, o Pampa e o Pantanal. A seguir, conheça cada bioma do Brasil. Amazônia Extensão aproximada: 4.196.943 quilômetros quadrados', 'passage_hash': '06d1938e1a8b96cdcdbc

In [18]:
passage_ids_from_queries = np.array([[r['passage_id'] for r in res] for res in results])


In [19]:
passage_ids_from_queries[0][0]

np.str_('clueweb22-pt0000-05-02747_0')

In [29]:
qrels.query(f'query_id == {0+1}').sort_values('score', ascending=False).head(10).shape

(10, 3)